 Imports & Configuration
- On charge les bibliothèques Python nécessaires.
- `pandas` : lire, manipuler et sauvegarder les CSV
- `os` : gérer les chemins de fichiers
- `re` : manipuler les chaînes de texte

In [11]:
import pandas as pd
import os
import re

# ── Chemins ──────────────────────────────────────────────────────────────────
# Modifiez INPUT_DIR pour pointer vers le dossier où sont vos CSV
INPUT_DIR  = "./tables"   # dossier contenant les CSV bruts Airtable
OUTPUT_DIR = "./clean_tables"      # dossier où seront sauvegardés les CSV nettoyés

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(" Configuration OK")
print(f"   Lecture depuis  : {os.path.abspath(INPUT_DIR)}")
print(f"   Écriture vers   : {os.path.abspath(OUTPUT_DIR)}")

 Configuration OK
   Lecture depuis  : c:\Maria\digitalPlanner\tables
   Écriture vers   : c:\Maria\digitalPlanner\clean_tables


Chargement de tous les CSV
- On charge tous les fichiers en mémoire d'un coup pour pouvoir les comparer entre eux.

Le paramètre `encoding='utf-8-sig'` gère le BOM (caractère invisible en début de fichier que Windows/Excel ajoute parfois).

In [12]:
# Chargement de chaque table
df_availability   = pd.read_csv(f"{INPUT_DIR}/MD_Availability-Grid view.csv",     encoding="utf-8-sig")
df_customer       = pd.read_csv(f"{INPUT_DIR}/MD_Customer-Grid view.csv",          encoding="utf-8-sig")
df_holidays       = pd.read_csv(f"{INPUT_DIR}/MD_Holidays-Grid view.csv",          encoding="utf-8-sig")
df_operations     = pd.read_csv(f"{INPUT_DIR}/MD_Operation-Grid view.csv",         encoding="utf-8-sig")
df_skills         = pd.read_csv(f"{INPUT_DIR}/MD_Skills-Grid view.csv",            encoding="utf-8-sig")
df_technicians    = pd.read_csv(f"{INPUT_DIR}/MD_Technicians-Grid view.csv",       encoding="utf-8-sig")
df_work_centers   = pd.read_csv(f"{INPUT_DIR}/MD_Work Centers-Grid view.csv",      encoding="utf-8-sig")
df_woo            = pd.read_csv(f"{INPUT_DIR}/Work Order Operations-Grid view.csv",encoding="utf-8-sig")
df_wo             = pd.read_csv(f"{INPUT_DIR}/Work Order-Grid view.csv",            encoding="utf-8-sig")
df_asset_crit     = pd.read_csv(f"{INPUT_DIR}/MD_Asset Criticality-Grid view.csv", encoding="utf-8-sig")
df_assets         = pd.read_csv(f"{INPUT_DIR}/MD_Assets-Grid view.csv",            encoding="utf-8-sig")

# Affichage d'un résumé : nombre de lignes et colonnes par table
tables = {
    "Availability"        : df_availability,
    "Customer"            : df_customer,
    "Holidays"            : df_holidays,
    "Operations"          : df_operations,
    "Skills"              : df_skills,
    "Technicians"         : df_technicians,
    "Work_Centers"        : df_work_centers,
    "Work_Order_Ops"      : df_woo,
    "Work_Order"          : df_wo,
    "Asset_Criticality"   : df_asset_crit,
    "Assets"              : df_assets,
}

print(f"{'Table':<22} {'Lignes':>8} {'Colonnes':>10}")
print("-" * 42)
for name, df in tables.items():
    print(f"{name:<22} {len(df):>8} {len(df.columns):>10}")

Table                    Lignes   Colonnes
------------------------------------------
Availability                  2          7
Customer                     62          4
Holidays                      0          7
Operations                   27          8
Skills                        4          3
Technicians                   7         21
Work_Centers                  3          6
Work_Order_Ops             1096         21
Work_Order                    1         12
Asset_Criticality             3          2
Assets                      145         15


Inspection visuelle des colonnes
-Avant de supprimer quoi que ce soit, on vérifie exactement quels noms de colonnes existent dans chaque table. Les noms dans Airtable peuvent différer légèrement de notre audit (espaces, majuscules, etc.).

In [13]:
for name, df in tables.items():
    print(f"\n{'='*10} {name} {'='*10}")
    for col in df.columns:
        print(f"   • {col}")


========== Availability ==========
   • Working Hour ID
   • Status
   • Work Hour Description
   • Start Hour
   • End Hour
   • Pause start
   • Pause end

========== Customer ==========
   • Customer ID
   • Status
   • Customer Description
   • MD_Assets

========== Holidays ==========
   • Holiday request day and time
   • Status
   • Requested by technician
   • Technician first and last name (from Requested by technician)
   • Requested for
   • Full or half day
   • Raison

========== Operations ==========
   • Operation ID
   • Operation key
   • Operation Description
   • Duration
   • Duration unit
   • MD_Skills_Possibilities
   • Work Order Operations
   • Predescesor Operation key

========== Skills ==========
   • Skill ID
   • Skill Description
   • MD_Technicians

========== Technicians ==========
   • Technician ID
   • Technicial status
   • Technician First Name
   • Technician Last Name
   • Technician first and last name
   • MD_Skills
   • Skill Description
   •

In [14]:
print(f"Nombre de colonnes : {len(df_customer.columns)}")
print(f"Colonnes : {list(df_customer.columns)}")

Nombre de colonnes : 4
Colonnes : ['Customer ID', 'Status', 'Customer Description', 'MD_Assets']


Renommage des colonnes (standardisation)
- Airtable exporte des noms de colonnes avec des espaces, des parenthèses, des accents... 
On les renomme en `snake_case` propre pour faciliter tout le reste du travail.

Exemple : `"Technician First Name"` → `"technician_first_name"`

In [15]:
# ── Table : Availability ──────────────────────────────────────────────────────
# On garde : ID, status, description, start_hour, end_hour, pause_start, pause_end
df_availability.columns = [
    "pk_working_hour_id",
    "status",
    "working_hour_description",
    "start_hour",
    "end_hour",
    "pause_start",
    "pause_end"
]

# ── Table : Customer ─────────────────────────────────────────────────────────
# On supprime la colonne MD_Assets (liste d'IDs dénormalisée — la relation
# existe déjà dans Assets via fk_customer)
df_customer.columns = ["pk_customer_id", "status", "customer_description", "_assets_ids"]
df_customer = df_customer.drop(columns=["_assets_ids"])

# ── Table : Holidays ─────────────────────────────────────────────────────────
# Problème structurel : la FK pointe sur un NOM au lieu d'un ID → on va corriger ça
if len(df_holidays) > 0:
    df_holidays.columns = [
        "holiday_request_time",
        "status",
        "fk_technician_id",                  # sera l'ID après correction
        "technician_name_raw",               # nom brut — sera supprimé après jointure
        "requested_for",
        "full_or_half_day",
        "reason"
    ]
else:
    print(" Holidays est vide — création d'un DataFrame vide avec le bon schéma")
    df_holidays = pd.DataFrame(columns=[
        "holiday_request_time", "status", "fk_technician_id",
        "requested_for", "full_or_half_day", "reason"
    ])

# ── Table : Operations ───────────────────────────────────────────────────────
# On supprime les colonnes de liaison dénormalisées (MD_Skills_Possibilities,
# Work Order Operations) — ces relations existent via FK dans d'autres tables
df_operations.columns = [
    "pk_operation_id",
    "operation_key",
    "operation_description",
    "duration",
    "duration_unit",
    "_skills_ids",          # liste dénormalisée → supprimée
    "_woo_ids",             # liste dénormalisée → supprimée
    "predecessor_operation_key"
]
df_operations = df_operations.drop(columns=["_skills_ids", "_woo_ids"])

# ── Table : Skills ───────────────────────────────────────────────────────────
# On supprime MD_Technicians (liste dénormalisée)
df_skills.columns = ["pk_skill_id", "skill_description", "_technicians_ids"]
df_skills = df_skills.drop(columns=["_technicians_ids"])

# ── Table : Technicians ──────────────────────────────────────────────────────
# NETTOYAGE MAJEUR :
# - Supprimer skill_description       (dupliqué depuis Skills)
# - Supprimer tous les champs availability (dupliqués depuis Availability)
# - Supprimer work_center_description (dupliqué depuis Work_Centers)
# - Conserver uniquement les FK vers les autres tables
df_technicians.columns = [
    "pk_technician_id",
    "technician_status",
    "technician_first_name",
    "technician_last_name",
    "technician_full_name",
    "fk_skill_id",
    "_skill_description_DUPLICATE",      # ← dupliqué de Skills → SUPPRIMÉ
    "fk_availability_id",
    "_availability_desc_DUPLICATE",      # ← dupliqué de Availability → SUPPRIMÉ
    "_start_hour_DUPLICATE",             # ← dupliqué de Availability → SUPPRIMÉ
    "_start_hour2_DUPLICATE",            # ← dupliqué de Availability → SUPPRIMÉ
    "_pause_start_DUPLICATE",            # ← dupliqué de Availability → SUPPRIMÉ
    "_pause_end_DUPLICATE",              # ← dupliqué de Availability → SUPPRIMÉ
    "address_street",
    "address_door",
    "address_post_code",
    "address_city",
    "fk_work_center_id",
    "_work_center_desc_DUPLICATE",       # ← dupliqué de Work_Centers → SUPPRIMÉ
    "_woo_ids",                          # liste dénormalisée → SUPPRIMÉE
    "_holidays_ids"                      # liste dénormalisée → SUPPRIMÉE
]
cols_to_drop_tech = [
    "_skill_description_DUPLICATE",
    "_availability_desc_DUPLICATE",
    "_start_hour_DUPLICATE",
    "_start_hour2_DUPLICATE",
    "_pause_start_DUPLICATE",
    "_pause_end_DUPLICATE",
    "_work_center_desc_DUPLICATE",
    "_woo_ids",
    "_holidays_ids"
]
df_technicians = df_technicians.drop(columns=cols_to_drop_tech)

# ── Table : Work_Centers ─────────────────────────────────────────────────────
# On supprime les colonnes dénormalisées (noms et compétences des techniciens)
# Ces infos sont disponibles via la FK fk_work_center_id dans Technicians
df_work_centers.columns = [
    "pk_work_center_id",
    "work_center_status",
    "work_center_description",
    "_technicians_ids",                  # liste dénormalisée → SUPPRIMÉE
    "_technician_names_DUPLICATE",       # dupliqué de Technicians → SUPPRIMÉ
    "_skill_desc_DUPLICATE"              # dupliqué de Skills → SUPPRIMÉ
]
df_work_centers = df_work_centers.drop(columns=[
    "_technicians_ids", "_technician_names_DUPLICATE", "_skill_desc_DUPLICATE"
])

# ── Table : Asset_Criticality ────────────────────────────────────────────────
# On supprime la liste d'IDs d'assets (relation inverse dénormalisée)
df_asset_crit.columns = ["pk_criticality_name", "_assets_ids"]
df_asset_crit = df_asset_crit.drop(columns=["_assets_ids"])

# ── Table : Assets ───────────────────────────────────────────────────────────
# NETTOYAGE MAJEUR :
# - Supprimer work_center_description (dupliqué de Work_Centers)
# - Supprimer customer_description    (dupliqué de Customer)
# - Supprimer Work Order              (liste dénormalisée)
df_assets.columns = [
    "pk_asset_id",
    "asset_status",
    "asset_description",
    "address_street",
    "address_door",
    "address_post_code",
    "address_city",
    "asset_region",
    "fk_work_center_id",
    "_work_center_desc_DUPLICATE",       # dupliqué de Work_Centers → SUPPRIMÉ
    "fk_customer_id",
    "_customer_desc_DUPLICATE",          # dupliqué de Customer → SUPPRIMÉ
    "_work_order_ids",                   # liste dénormalisée → SUPPRIMÉE
    "fk_criticality",
    "created_at"
]
df_assets = df_assets.drop(columns=[
    "_work_center_desc_DUPLICATE",
    "_customer_desc_DUPLICATE",
    "_work_order_ids"
])

# ── Table : Work_Order ───────────────────────────────────────────────────────
# NETTOYAGE MAJEUR :
# - Supprimer asset_description       (dupliqué de Assets)
# - Supprimer work_center_description (dupliqué de Work_Centers)
# - Supprimer customer_description    (dupliqué de Customer)
df_wo.columns = [
    "pk_work_order_id",
    "work_order_type",
    "_woo_ids",                          # liste dénormalisée → SUPPRIMÉE
    "breakdown",
    "priority",
    "order_basic_start_date",
    "order_basic_end_date",
    "last_inspection_date",
    "fk_asset_id",
    "_asset_desc_DUPLICATE",             # dupliqué de Assets → SUPPRIMÉ
    "_work_center_desc_DUPLICATE",       # dupliqué de Work_Centers → SUPPRIMÉ
    "_customer_desc_DUPLICATE"           # dupliqué de Customer → SUPPRIMÉ
]
df_wo = df_wo.drop(columns=[
    "_woo_ids",
    "_asset_desc_DUPLICATE",
    "_work_center_desc_DUPLICATE",
    "_customer_desc_DUPLICATE"
])

# ── Table : Work_Order_Operations ────────────────────────────────────────────
# NETTOYAGE MAJEUR :
# - Supprimer technician_first_name   (dupliqué de Technicians)
# - Supprimer asset_work_center       (dupliqué de Work_Centers via Assets)
# - Supprimer work_order_id_auto      (doublon de work_order_id)
df_woo.columns = [
    "pk_woo_id",
    "fk_work_order_id",
    "_work_order_id_auto_DUPLICATE",     # doublon → SUPPRIMÉ
    "status",
    "fk_operation_id",
    "operation_key",
    "operation_description",
    "duration",
    "duration_unit",
    "predecessor_operation_key",
    "fk_required_skill_id",
    "_required_skill_desc",              # utile pour lisibilité — on garde
    "order_basic_start_date",
    "order_basic_end_date",
    "operation_scheduled_start",
    "operation_scheduled_end",
    "confirmed_work",
    "confirmed_work_unit",
    "_asset_work_center_DUPLICATE",      # dupliqué via Assets → SUPPRIMÉ
    "fk_assigned_technician_id",
    "_technician_name_DUPLICATE"         # dupliqué de Technicians → SUPPRIMÉ
]
df_woo = df_woo.drop(columns=[
    "_work_order_id_auto_DUPLICATE",
    "_asset_work_center_DUPLICATE",
    "_technician_name_DUPLICATE"
])

print(" Renommage et suppression des colonnes dupliquées terminés")

 Holidays est vide — création d'un DataFrame vide avec le bon schéma
 Renommage et suppression des colonnes dupliquées terminés


---
##  Correction du problème structurel : Holidays
** Dans la table Holidays, la FK vers le technicien utilise un **nom** (`Luc Dupuis`) au lieu d'un **ID** (`6`). C'est dangereux : si un technicien change de nom, le lien est cassé.

On corrige en faisant une **jointure** : on cherche l'ID correspondant au nom dans la table Technicians.

In [16]:
if len(df_holidays) > 0 and "technician_name_raw" in df_holidays.columns:
    # Créer un dictionnaire nom → ID depuis la table Technicians
    name_to_id = dict(zip(
        df_technicians["technician_full_name"],
        df_technicians["pk_technician_id"]
    ))
    
    # Remplacer le nom brut par l'ID correspondant
    df_holidays["fk_technician_id"] = df_holidays["technician_name_raw"].map(name_to_id)
    
    # Supprimer la colonne nom brut — elle ne sert plus
    df_holidays = df_holidays.drop(columns=["technician_name_raw"])
    
    # Vérifier qu'aucun ID n'est manquant (NaN = nom non trouvé dans Technicians)
    missing = df_holidays["fk_technician_id"].isna().sum()
    if missing > 0:
        print(f"  {missing} lignes Holidays sans technicien correspondant")
    else:
        print(" Holidays : FK corrigée — tous les noms ont un ID correspondant")
else:
    print("ℹ  Holidays est vide — rien à corriger")

ℹ  Holidays est vide — rien à corriger


# Découpage de operation_key en 3 champs distincts 

In [ ]:

# Fonction qui parse "Corrective-Breakdown-1" → ('Corrective', 'Breakdown', 1)
def parse_operation_key(key):
    if pd.isna(key) or key == "":
        return None, None, None
    parts = str(key).split("-")
    op_type     = parts[0] if len(parts) > 0 else None   # Corrective / Preventive
    op_subtype  = parts[1] if len(parts) > 1 else None   # Breakdown / Low / Meca...
    op_order    = int(parts[2]) if len(parts) > 2 and parts[2].isdigit() else None
    return op_type, op_subtype, op_order

# Appliquer sur la table Operations
df_operations[["operation_type", "operation_subtype", "operation_order"]] = df_operations["operation_key"].apply(
    lambda k: pd.Series(parse_operation_key(k))
)

# Appliquer sur WOO aussi
df_woo[["operation_type", "operation_subtype", "operation_order"]] = df_woo["operation_key"].apply(
    lambda k: pd.Series(parse_operation_key(k))
)

# Vérification
print(df_operations[["operation_key", "operation_type", "operation_subtype", "operation_order"]].to_string())


              operation_key operation_type operation_subtype  operation_order
0    Corrective-Breakdown-1     Corrective         Breakdown                1
1    Corrective-Breakdown-2     Corrective         Breakdown                2
2    Corrective-Breakdown-3     Corrective         Breakdown                3
3    Corrective-Breakdown-4     Corrective         Breakdown                4
4    Corrective-Breakdown-5     Corrective         Breakdown                5
5    Corrective-Breakdown-6     Corrective         Breakdown                6
6    Corrective-Breakdown-7     Corrective         Breakdown                7
7    Corrective-Breakdown-8     Corrective         Breakdown                8
8    Corrective-Breakdown-9     Corrective         Breakdown                9
9   Corrective-Breakdown-10     Corrective         Breakdown               10
10         Corrective-Low-1     Corrective               Low                1
11         Corrective-Low-2     Corrective               Low    

Vérification de l'intégrité des FK
 Avant de sauvegarder, on vérifie que toutes les FK pointent vers des IDs qui existent réellement dans leurs tables de référence.

Exemple : chaque `fk_work_center_id` dans Technicians doit exister dans `pk_work_center_id` de Work_Centers.

In [18]:
def check_fk(child_df, child_col, parent_df, parent_col, label):
    """
    Vérifie qu'une FK ne pointe pas vers un ID inexistant.
    child_df[child_col]  → les valeurs FK à vérifier
    parent_df[parent_col] → les IDs de référence
    """
    # On ignore les valeurs nulles (FK optionnelles)
    child_vals  = set(child_df[child_col].dropna().astype(str))
    parent_vals = set(parent_df[parent_col].dropna().astype(str))
    orphans = child_vals - parent_vals
    if orphans:
        print(f" {label}: {len(orphans)} valeur(s) orpheline(s) → {list(orphans)[:5]}")
    else:
        print(f" {label}: OK")

print("── Vérification des Foreign Keys ───────────────────────────────")
check_fk(df_technicians, "fk_skill_id",         df_skills,       "pk_skill_id",         "Technicians → Skills")
check_fk(df_technicians, "fk_availability_id",  df_availability, "pk_working_hour_id",  "Technicians → Availability")
check_fk(df_technicians, "fk_work_center_id",   df_work_centers, "pk_work_center_id",   "Technicians → Work_Centers")
check_fk(df_assets,      "fk_work_center_id",   df_work_centers, "pk_work_center_id",   "Assets → Work_Centers")
check_fk(df_assets,      "fk_customer_id",      df_customer,     "pk_customer_id",      "Assets → Customer")
check_fk(df_assets,      "fk_criticality",      df_asset_crit,   "pk_criticality_name", "Assets → Asset_Criticality")
check_fk(df_wo,          "fk_asset_id",         df_assets,       "pk_asset_id",         "Work_Order → Assets")
check_fk(df_woo,         "fk_work_order_id",    df_wo,           "pk_work_order_id",    "WOO → Work_Order")
check_fk(df_woo,         "fk_operation_id",     df_operations,   "pk_operation_id",     "WOO → Operations")

── Vérification des Foreign Keys ───────────────────────────────
 Technicians → Skills: OK
 Technicians → Availability: OK
 Technicians → Work_Centers: OK
 Assets → Work_Centers: OK
 Assets → Customer: OK
 Assets → Asset_Criticality: OK
 Work_Order → Assets: OK
 WOO → Work_Order: 176 valeur(s) orpheline(s) → ['114', '12', '23', '33', '78']
 WOO → Operations: OK
